In [3]:
import os
import sys
import ast
import json
import time
import shutil
import pickle
import argparse

import numpy as np
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer

PROJECT_DIR = "/net/dali/home/barton/dhw28/popDMS/esmDMS"
sys.path.insert(0, PROJECT_DIR)

from esmdmsfunctions import CODON2AA

DATA_DIR = os.path.join(PROJECT_DIR, "data", "raw_data")
HOME_SEQ_FOLDER = os.path.join(PROJECT_DIR, "data", "sequence_data")

In [ ]:
# Sample file format:

data_dir = "/Users/dylanwells/popDMS/esmDMS/data/raw_data/"

tpor_filepath = data_dir + "TpoR_nucleotide_counts.csv"
tpor_reference_filepath = data_dir + "TpoR_reference_sequence.dat"

'''
accession,hgvs_nt,hgvs_splice,hgvs_pro,Replicate_A_c_0,Replicate_A_c_1,Replicate_B_c_0,Replicate_B_c_1,Replicate_C_c_0,Replicate_C_c_1,Replicate_D_c_0,Replicate_D_c_1,Replicate_E_c_0,Replicate_E_c_1,Replicate_F_c_0,Replicate_F_c_1
urn:mavedb:00000043-a-1#1,c.93T>G,NA,p.Phe31Leu,5.0,19.0,5.0,7.0,5.0,22.0,5.0,18.0,5.0,30.0,5.0,20.0
urn:mavedb:00000043-a-1#2,c.93T>A,NA,p.Phe31Leu,71.0,195.0,71.0,104.0,71.0,209.0,71.0,192.0,71.0,232.0,71.0,208.0
urn:mavedb:00000043-a-1#3,c.92T>G,NA,p.Phe31Cys,34.0,51.0,34.0,39.0,34.0,68.0,34.0,51.0,34.0,65.0,34.0,84.0
urn:mavedb:00000043-a-1#4,c.92T>C,NA,p.Phe31Ser,70.0,49.0,70.0,35.0,70.0,55.0,70.0,62.0,70.0,40.0,70.0,55.0
urn:mavedb:00000043-a-1#5,c.92T>A,NA,p.Phe31Tyr,75.0,54.0,75.0,36.0,75.0,62.0,75.0,57.0,75.0,64.0,75.0,52.0
urn:mavedb:00000043-a-1#6,c.91T>G,NA,p.Phe31Val,111.0,99.0,111.0,57.0,111.0,93.0,111.0,146.0,111.0,99.0,111.0,137.0
urn:mavedb:00000043-a-1#7,c.91T>C,NA,p.Phe31Leu,45.0,33.0,45.0,18.0,45.0,30.0,45.0,30.0,45.0,29.0,45.0,28.0
urn:mavedb:00000043-a-1#8,c.91T>A,NA,p.Phe31Ile,87.0,85.0,87.0,22.0,87.0,110.0,87.0,77.0,87.0,93.0,87.0,83.0
'''

In [3]:
import pandas as pd

test = pd.read_pickle("/net/dali/home/barton/dhw28/popDMS/esmDMS/data/sequence_data/TpoR_embeddings.pkl")

In [4]:
test

,ProteinSequence,PreNums,PostNums,Embeddings
0,AETAWISLVTALHLVLGLSAVLGLLLLRWQF,"[38, 38, 38, 38, 38, 38]","[101, 22, 25, 49, 42, 73]","[[0.04104695, -0.022587178, -0.020405274, -0.0..."
1,IETAWISLVTALHLVLGLNAVLGLLLLRWQF,"[1, 1, 1, 1, 1, 1]","[4, 4, 6, 5, 6, 8]","[[0.038780514, -0.020099387, -0.02574463, -0.0..."
2,IETAWISLVTALHLVLGLSAVLGLLLLRKQF,"[1, 1, 1, 1, 1, 1]","[6, 3, 7, 6, 5, 6]","[[0.04167486, -0.01922943, -0.018841147, -0.02..."
3,IETAWISLVTALHLVLGLSAVLGLLLLRWQF,"[54, 54, 54, 54, 54, 54]","[36, 28, 27, 59, 39, 82]","[[0.03839396, -0.019617615, -0.020410158, -0.0..."
4,NETAWISLVTALHLSLGLSAVLGLLLLRWQF,"[1, 1, 1, 1, 1, 1]","[7, 6, 10, 9, 18, 11]","[[0.041083574, -0.016726686, -0.015839031, -0...."
...,...,...,...,...
1128,TVTAWISLVTALHLVLGLNAVLGLLLLRWQF,"[1, 1, 1, 1, 1, 1]","[12, 12, 19, 29, 17, 64]","[[0.039902955, -0.01961792, -0.027980959, -0.0..."
1129,TVTAWISLVTALHLVLGLSAVLGLLLLRSQF,"[4, 4, 4, 4, 4, 4]","[10, 4, 6, 11, 5, 21]","[[0.041526485, -0.016229453, -0.0159668, -0.01..."
1130,TVTAWISLVTALHLVLGLSAVLGLLLLRWQF,"[4034, 4034, 4034, 4034, 4034, 4034]","[603, 395, 1079, 1347, 741, 3300]","[[0.039516397, -0.01913615, -0.022646485, -0.0..."
1131,TWTAWISLVTALHLVLGLSAVLGLLLLRWQF,"[1390, 1390, 1390, 1390, 1390, 1390]","[3384, 52, 376, 201, 1188, 178]","[[0.03737935, -0.019368388, -0.0246403, -0.016..."
